# Loading Source Datasets for Featurization
This notebook illustrates how the featurization module would load the input data files for the featurization stage. All that is needed is the config files for the workspace. This encodes the meta-data and file locations for the output files. This is illustrated for all the data files that we have in `dd-parser-cleaner`.



In [4]:
from pathlib import Path
import yaml
import pandas as pd
from IPython.display import display, Markdown


In [5]:
def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'notebooks' / 'example_configs').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not locate the repository root containing notebooks/example_configs.')

def load_example_config(config_file: Path) -> dict:
    with config_file.open('r', encoding='utf-8') as fh:
        return yaml.safe_load(fh) or {}

def resolve_cleaner_paths(config: dict) -> dict:
    working_dir = Path(config.get('working_dir', '.')).resolve()
    cleaner = config.get('cleaner', {})
    dd_dir = Path(cleaner.get('dd_cleaner_output_dir', 'dd_cleaner'))
    base_dir = working_dir / 'data' / dd_dir
    return {
        'raw_data': base_dir / cleaner.get('raw_dataset_file', ''),
        'cleaned_data': base_dir / cleaner.get('clean_output_filename', ''),
        'user_cleaned_data': base_dir / cleaner.get('user_cleaned_output_filename', ''),
        'metadata_table': base_dir / cleaner.get('metadata_table_filename', ''),
    }


In [6]:
repo_root = find_repo_root()
example_config_dir = repo_root / 'notebooks' / 'example_configs'
config_files = ['sba_config.yaml', 'olist_config.yaml', 'itsm_config.yaml']

for config_name in config_files:
    config_path = example_config_dir / config_name
    config = load_example_config(config_path)

    display(Markdown(f'## {config_name}'))

    display(Markdown('### Dataset bootstrap metadata'))
    metadata_summary = {
        'working_dir': config.get('working_dir'),
        'dataset_type': config.get('dataset_type'),
        'dataset_id': config.get('dataset_id'),
        'subject': config.get('subject'),
        'graph_type': config.get('graph_type'),
        'wide_short_homogeneous': config.get('wide_short_homogeneous'),
        'notes': config.get('notes'),
    }
    display(pd.DataFrame([metadata_summary]))

    display(Markdown('### Cleaner artifact filenames from config'))
    artifact_filenames = [
        ('Raw dataset file', config.get('cleaner', {}).get('raw_dataset_file')),
        ('Cleaned dataset file', config.get('cleaner', {}).get('clean_output_filename')),
        ('User cleaned dataset file', config.get('cleaner', {}).get('user_cleaned_output_filename')),
        ('Metadata authority table', config.get('cleaner', {}).get('metadata_table_filename')),
        ('DD cleaner output dir', config.get('cleaner', {}).get('dd_cleaner_output_dir'))
    ]
    display(pd.DataFrame(artifact_filenames, columns=['Artifact', 'Filename']))

    resolved_paths = resolve_cleaner_paths(config)
    display(Markdown('### Resolved cleaner artifact paths'))
    display(pd.DataFrame([
        {'Artifact': key, 'Path': str(path), 'Exists': path.exists()}
        for key, path in resolved_paths.items()
    ]))

    display(Markdown('---'))


## sba_config.yaml

### Dataset bootstrap metadata

,working_dir,dataset_type,dataset_id,subject,graph_type,wide_short_homogeneous,notes
0,/home/rajiv/programming/dd-parser-cleaner/tests,cross-sectional,sba_loans_raw,loan,None,None,Generated by dataset bootstrapping.


### Cleaner artifact filenames from config

,Artifact,Filename
0,Raw dataset file,sba_loans_raw.csv
1,Cleaned dataset file,sba_loans_raw_clean.csv
2,User cleaned dataset file,sba_loans_raw_user_cleaned.csv
3,Metadata authority table,sba_loans_raw_metadata_table.csv
4,DD cleaner output dir,dd_cleaner


### Resolved cleaner artifact paths

... (14 outputs left out)

,Artifact,Filename
0,Raw dataset file,itsm_group_analysis.csv
1,Cleaned dataset file,itsm_group_analysis_clean.csv
2,User cleaned dataset file,itsm_group_analysis_user_cleaned.csv
3,Metadata authority table,itsm_group_analysis_metadata_table.csv
4,DD cleaner output dir,dd_cleaner


### Resolved cleaner artifact paths

,Artifact,Path,Exists
0,raw_data,/home/rajiv/programming/dd-parser-cleaner/test...,False
1,cleaned_data,/home/rajiv/programming/dd-parser-cleaner/test...,True
2,user_cleaned_data,/home/rajiv/programming/dd-parser-cleaner/test...,False
3,metadata_table,/home/rajiv/programming/dd-parser-cleaner/test...,False


---

## Confirming cleaned data extraction

The cleaned dataset filename is exposed under `cleaner.user_cleaned_output_filename` in each config.
The notebook also resolves the full path relative to the configured `working_dir` and `dd_cleaner_output_dir`,
so this proves the cleaned data file can be extracted from the config.
